#### 1. Data formatting to tensors

In [3]:
#0. Import libaries 
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn import metrics
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader
from protgen.data.make_data import IGNORE_INDEX

In [4]:
#1. Data 
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Shifted shape: {y_train_shifted.shape}")

# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")


Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22,
        16,  8, 11, 19, 10, 14, 12, 

In [5]:
import torch
import torch.nn.functional as F
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

print(f"vocab ({train_data.vocab_size} tokens): {train_data.vocab}")
print(f"families: {train_data.fam_to_idx}")
print(f"padding is ignored in the loss with ignore_index={IGNORE_INDEX}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

sequence = train_data.df["sequence"][0]
tokens = train_data.encode(sequence)
print(sequence)
print(tokens)
print()
print(f"encode / decode round-trips: {train_data.decode(tokens) == sequence}")

x, x_shifted, y, length = next(iter(train_loader))
print(f"batch: x {tuple(x.shape)} {x.dtype}, x_shifted {tuple(x_shifted.shape)}, y {tuple(y.shape)}")

# Generative: one distribution over the vocabulary per position.
logits_gen = torch.randn(x.shape[0], train_data.vocab_size, train_data.max_length)
print(f"generative loss:     {F.cross_entropy(logits_gen, x_shifted, ignore_index=IGNORE_INDEX):.4f}")

vocab (23 tokens): {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, 'A': 3, 'C': 4, 'D': 5, 'E': 6, 'F': 7, 'G': 8, 'H': 9, 'I': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'V': 20, 'W': 21, 'Y': 22}
families: {'AcyP': 0, 'HPr': 1, 'HU': 2}
padding is ignored in the loss with ignore_index=-1
MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHSAKSIMDIMSLSIRPGTKFYIKAEGIDEVQAIQLIQDYLYKEKLINY
tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])

enc

In [6]:
print(x)
print(x.shape)
print(x.dtype)
print(train_data[3])

tensor([[ 1, 13,  3,  ...,  0,  0,  0],
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 18,  ...,  0,  0,  0],
        ...,
        [ 1, 13, 18,  ...,  0,  0,  0],
        [ 1, 13, 15,  ...,  0,  0,  0],
        [ 1, 13, 15,  ...,  0,  0,  0]])
torch.Size([32, 129])
torch.int64
(tensor([ 1, 13, 14, 11, 19,  6, 12, 10, 12, 17, 10,  3,  6,  6, 19,  8,  7, 18,
        17, 11, 14, 20,  6, 12,  8, 12, 14,  3,  3, 10, 14, 12, 10, 19, 11,  3,
        12, 19,  6,  6,  6, 11, 20, 16, 10, 20,  8,  7,  8,  3,  7,  6, 19, 11,
        20, 17,  3,  6, 17, 13,  8, 17, 14, 15, 11, 19,  8,  6,  5, 10, 15, 10,
        15,  3, 19, 11, 20, 15, 20,  7, 11,  3,  8, 11,  3, 12, 11,  5,  3, 20,
         5, 16,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0]), tensor([13, 14, 11, 19,  6, 12, 10, 12, 17, 10,  3,  6,  6, 19,  8,  7, 18, 17,
        11, 14, 20,  6, 12,  8, 12, 14,  3,  3, 1

#### 2. Model

In [13]:
import torch
from torch import nn

#Init function
class MLP(nn.Module):
    "Initial multilayer perceptron"
    def __init__(self, hidden_layers, hidden_layers2, embedding_dim, batch_size):
        super(MLP, self).__init__()

        #Stablished parameters
        voc = train_data.vocab_size
        activation_fn = nn.ReLU

        self.flatten = nn.Flatten()
        input_dim = train_data.max_length * embedding_dim
        
        #Embedding variables
        self.embedding = nn.Embedding(num_embeddings=voc, embedding_dim=embedding_dim, padding_idx=IGNORE_INDEX)

        #MLP structure
        self.net = nn.Sequential(
            #Initial shape is 32
            nn.Linear(input_dim,hidden_layers),
            activation_fn(),
            nn.Linear(hidden_layers,hidden_layers2),
            activation_fn(),
            #Desired output is vocab_size
            nn.Linear(hidden_layers2,train_data.vocab_size))

    def forward(self, x):
        #Embed
        step1 = self.embedding(x)
        #Flatten
        step1 = self.flatten(step1)
        #Run model
        step2 = self.net(step1)
        #Return most likely result
        return step2

#### 3. Train (heavily inspired by 4.2 CNN CIFAR-10)

#3.1 Define loss and optimizer

In [16]:
model = MLP(hidden_layers=64, hidden_layers2=32, embedding_dim=32, batch_size=32)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [17]:
#Start variables
step = 0
model.train()
train_accuracies = []
validation_accuracies = []

#Defined hyperparameters
num_epochs = 5
validation_every_steps = 500
batch_size = 32 

#Data loaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, drop_last=False)

#Start training loop
for epoch in range(num_epochs):
    train_accuracies_batches = []
    
    for x, x_shifted, y, length in train_loader:
        
        #Input == output
        output = model(x)
        print(f"output shape: {output.shape}, x_shifted shape: {x_shifted.shape}")
        #Calculate loss of our prediction 
        loss = loss_fn(output, x_shifted)
        #Clean up gradients
        optimizer.zero_grad()    
        #New gradients and optimization
        loss.backward()
        optimizer.step()
        step += 1
        
        #Pred
        predictions = output.max(1)[1]
        #Need to ignore padding tokens for acc
        fl = x_shifted != IGNORE_INDEX
        train_accuracies_batches.append(metrics.accuracy_score(x_shifted[fl], predictions[fl]))
        
        if step % validation_every_steps == 0:
            #Append average training accuracy to list.
            train_accuracies.append(np.mean(train_accuracies_batches))     
            train_accuracies_batches = []
        
            #Compute accuracies on validation set.
            validation_accuracies_batches = []
            with torch.no_grad():
                model.eval()
                for x, x_shifted, y, length in validation_loader:
                    output = model(x)
                    loss = loss_fn(output, x_shifted)

                    predictions = output.max(1)[1]
                    fl = x_shifted != IGNORE_INDEX

                    #Sum correct predictions CORRECT
                    right_pred = (predictions[fl] == x_shifted[fl]).sum().item()
                    total_pred = fl.sum().item()
                model.train()
                
            #Append average validation accuracy to list.
            validation_accuracies.append(right_pred / total_pred)
     
            print(f"Step {step:<5}   training accuracy: {train_accuracies[-1]}")
            print(f"             validation accuracy: {validation_accuracies[-1]}")

print("Finished training.")

output shape: torch.Size([32, 23]), x_shifted shape: torch.Size([32, 129])


RuntimeError: 0D or 1D target tensor expected, multi-target not supported